In [ ]:
import sys; sys.path.append('..')
import MeshFEM, mesh, mesh_energy, param_utils, viewer, benchmark
import numpy as np
import sim_utils

import matplotlib
from matplotlib import pyplot as plt
import visualization

import newton_flow

In [ ]:
np.set_printoptions(linewidth=1000, edgeitems=1000)

In [ ]:
import parallelism
parallelism.set_max_num_tbb_threads(1)

In [ ]:
m = param_utils.load('../../models/lucy.msh.xz')
# m = param_utils.load('../../models/cow2Disc.msh')
# m = param_utils.load('../../models/hilbert_curve.msh.xz')
tutte_uv = param_utils.tutteInitialization(m)
v = mesh_energy.NodalVars(m, 2)
v.setVars(tutte_uv.ravel())
m_2d = mesh.Mesh(np.zeros_like(tutte_uv), m.elements())
m_2d.reembedElements(m.vertices())
nf = newton_flow.symmetric_dirichlet(m_2d, v)

In [ ]:
import fast_newton_flow

In [ ]:
fnf = fast_newton_flow.symmetric_dirichlet(m_2d, v)

In [ ]:
nf.projectionSmoothingEpsilon = 0 # 1e-4 # 1e-8

In [ ]:
nf.objectiveAtVars(v.getVars())

In [ ]:
fnf.objective()

In [ ]:
import py_newton_optimizer
prob = py_newton_optimizer.NewtonMultiobjectiveProblem(v, [nf])

In [ ]:
# Nullspace pinning strategy
FIX_VARS = False
if FIX_VARS:
    # fv = sim_utils.getBBoxVars(m_rest, sim_utils.BBoxFace.MIN_X)
    # prob.setFixedVars(fv)
    import elastic_solid, energy
    es = elastic_solid.ElasticSolid(m_rest, energy.CommonNeoHookeanYoungPoisson(2, 1, 0.3))
    es.setDeformedPositions(m_defo.vertices())
    pin_vars, _ = es.prepareRigidMotionPins()
    v.setVars(es.getVars())
    prob.setFixedVars(pin_vars)
else:
    # prob.hessianShift = 1e-5
    # nf.elementHessianShift = 1e-8
    # nf.elementHessianShift = 1e-5
    # fnf.elementHessianShift = 1e-5
    prob.hessianShift = 1e-8
    prob.useRelativeHessianShift = False

In [ ]:
import newton_flow_utils

In [ ]:
constant_speed = True
always_project = True

In [ ]:
opt = prob.optimizer()
opt.options.hessianProjectionController.startWithProjectionActive = False
opt.options.hessianProjectionController.numProjectionStepsBeforeDisable = 1
opt.options.hessianProjectionController.numConsecutiveIndefiniteStepsBeforeEnable = 0
if always_project: opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAlways()

In [ ]:
# opt.options.factorizer = opt.options.factorizer.CatamariAMD

In [ ]:
# opt.options.factorizer = opt.options.factorizer.CHOLMOD

In [ ]:
# opt.options.factorizer = opt.options.factorizer.Accelerate

In [ ]:
benchmark.reset()
opt.options.niter = 20
opt.optimize()
opt.update_factorizations()
benchmark.report()

In [ ]:
benchmark.reset()
d = opt.newton_step()
benchmark.report()

In [ ]:
max_degree = 3

# Validation

In [ ]:
arclen = False
projectHessian = True

In [ ]:
def relerror(a, b):
    return np.linalg.norm(a - b) / np.linalg.norm(a)

In [ ]:
def relerror_centered(a, b):
    a = a.reshape(-1, 2).copy()
    b = b.reshape(-1, 2).copy()
    a -= np.mean(a, axis=0)
    b -= np.mean(b, axis=0)
    return np.linalg.norm(a - b) / np.linalg.norm(a)

In [ ]:
nf.remove_rigid_translation = True

In [ ]:
benchmark.reset()
fcoeffs = fnf.computeTaylorCoefficients(opt.hessian_factorization, d, projectHessian = projectHessian, degree=max_degree, arclen=arclen)

if arclen:
    coeffs = nf.computeTaylorCoefficientsArclen(opt.hessian_factorization, projectHessian = projectHessian, degree=max_degree) # warm up
    c = np.linalg.norm(fcoeffs[0])
    coeffs = [c**(i + 1) * x_i for i, x_i in enumerate(coeffs)]
else:
    coeffs = nf.computeTaylorCoefficients(opt.hessian_factorization, projectHessian = projectHessian, degree=max_degree) # warm up
# benchmark.report()

In [ ]:
# Note: the following are expected not to match in the case of arclength parametrization due to rescaling.
print(relerror(fnf.neg_delta_g, nf.neg_delta_g))
print(relerror_centered(fnf.neg_delta_g, nf.neg_delta_g))

In [ ]:
np.array([np.linalg.norm(fc) for fc in fcoeffs])

In [ ]:
np.array([np.linalg.norm(fc) for fc in coeffs])

In [ ]:
for i in range(max_degree):
    print(relerror(fcoeffs[i], coeffs[i]))

In [ ]:
for i in range(max_degree):
    print(relerror_centered(fcoeffs[i], coeffs[i]))

# Benchmarking

In [ ]:
fcoeffs = fnf.computeTaylorCoefficients(opt.hessian_factorization, d, projectHessian = False, degree=max_degree, arclen=True)

In [ ]:
import parallelism
parallelism.set_max_num_tbb_threads(14)

In [ ]:
max_degree = 20

In [ ]:
coeffs = nf.computeTaylorCoefficients(opt.hessian_factorization, projectHessian = True, degree=max_degree) # warm up
benchmark.reset()
coeffs = nf.computeTaylorCoefficients(opt.hessian_factorization, projectHessian = True, degree=max_degree)
benchmark.report()
aos_times = [benchmark.totalTime(f'order {i}$') for i in range(1, max_degree+1)]

In [ ]:
coeffs = nf.computeTaylorCoefficientsArclen(opt.hessian_factorization, projectHessian = True, degree=max_degree) # warm up
benchmark.reset()
coeffs = nf.computeTaylorCoefficientsArclen(opt.hessian_factorization, projectHessian = True, degree=max_degree)
benchmark.report()
aos_times = [benchmark.totalTime(f'order {i}$') for i in range(1, max_degree+1)]

In [ ]:
coeffs = nf.computeTaylorCoefficientsArclen(opt.hessian_factorization, projectHessian = False, degree=max_degree) # warm up
benchmark.reset()
coeffs = nf.computeTaylorCoefficientsArclen(opt.hessian_factorization, projectHessian = False, degree=max_degree)
benchmark.report()
aos_times = [benchmark.totalTime(f'order {i}$') for i in range(1, max_degree+1)]

In [ ]:
import parallelism
parallelism.set_max_num_tbb_threads(14)

In [ ]:
max_degree=20

In [ ]:
fcoeffs = fnf.computeTaylorCoefficients(opt.hessian_factorization, d, projectHessian = False, degree=max_degree, arclen=True) # warm up
benchmark.reset()
# max_degree = 100
fcoeffs = fnf.computeTaylorCoefficients(opt.hessian_factorization, d, projectHessian = False, degree=max_degree, arclen=True)
benchmark.report()
soa_times = [benchmark.totalTime(f'P upgrade {i}$') for i in range(2, max_degree+1)]

In [ ]:
nts = [1, 2, 4, 8, 10, 14]
thread_times = []
for nt in nts:
    parallelism.set_max_num_tbb_threads(nt)
    benchmark.reset()
    fcoeffs = fnf.computeTaylorCoefficients(opt.hessian_factorization, d, projectHessian = True, degree=max_degree, arclen=False)
    thread_times.append([benchmark.totalTime(f'P upgrade {i}$') for i in range(2, max_degree+1)])

In [ ]:
for nt, t in zip(nts, thread_times):
    plt.plot(t, label=nt)
plt.legend()

In [ ]:
proj = True

In [ ]:
parallelism.set_max_num_tbb_threads(14)

In [ ]:
fcoeffs = fnf.computeTaylorCoefficients(opt.hessian_factorization, d, projectHessian = proj, degree=max_degree, arclen=True) # warm up
benchmark.reset()
# max_degree = 100
fcoeffs = fnf.computeTaylorCoefficients(opt.hessian_factorization, d, projectHessian = proj, degree=max_degree, arclen=True)
benchmark.report()
soa_times = [benchmark.totalTime(f'P upgrade {i}$') for i in range(2, max_degree+1)]

In [ ]:
max_degree = 2

In [ ]:
fcoeffs = fnf.computeTaylorCoefficients(opt.hessian_factorization, d, projectHessian = proj, degree=max_degree, arclen=True) # warm up
benchmark.reset()
# max_degree = 100
fcoeffs = fnf.computeTaylorCoefficients(opt.hessian_factorization, d, projectHessian = proj, degree=max_degree, arclen=True)
benchmark.report()
soa_times = [benchmark.totalTime(f'P upgrade {i}$') for i in range(2, max_degree+1)]

In [ ]:
proj = False

In [ ]:
fcoeffs = fnf.computeTaylorCoefficients(opt.hessian_factorization, d, projectHessian = proj, degree=max_degree, arclen=True) # warm up
benchmark.reset()
# max_degree = 100
fcoeffs = fnf.computeTaylorCoefficients(opt.hessian_factorization, d, projectHessian = proj, degree=max_degree, arclen=True)
benchmark.report()

In [ ]:
fcoeffs = fnf.computeTaylorCoefficients(opt.hessian_factorization, d, projectHessian = proj, degree=max_degree, arclen=True) # warm up
benchmark.reset()
# max_degree = 100
fcoeffs = fnf.computeTaylorCoefficients(opt.hessian_factorization, d, projectHessian = proj, degree=max_degree, arclen=True)
benchmark.report()

# Benchmarking of Hessian Projection Slicing

In [ ]:
max_degree = 20

In [ ]:
proj = True

In [ ]:
tgt_pct = 0.25 # fraction of elements to eanble projection on
mask = nf.elementHessianMinimumEigenvalues() <= 0
while np.sum(mask) > tgt_pct * fnf.numElements():
    for i in range(100):
        mask[int(np.random.uniform(low=0, high=fnf.numElements()))] = False

In [ ]:
fnf.elementHessianProjectionMasks = mask

In [ ]:
fcoeffs = fnf.computeTaylorCoefficients(opt.hessian_factorization, d, projectHessian = proj, degree=max_degree, arclen=True) # warm up
benchmark.reset()
benchmark.start_timer_section('coeffs')
# max_degree = 100
fnf.initCoefficients(d, projectHessian = proj, arclen=True)
fnf.upgradeToDegree(opt.hessian_factorization, max_degree)
benchmark.stop_timer_section('coeffs')
benchmark.report()

In [ ]:
proj = False

In [ ]:
fcoeffs = fnf.computeTaylorCoefficients(opt.hessian_factorization, d, projectHessian = proj, degree=max_degree, arclen=True) # warm up
benchmark.reset()
benchmark.start_timer_section('coeffs')
# max_degree = 100
fnf.initCoefficients(d, projectHessian = proj, arclen=True)
fnf.upgradeToDegree(opt.hessian_factorization, max_degree)
benchmark.stop_timer_section('coeffs')
benchmark.report()

In [ ]:
benchmark.pieChart('upgradeToDegree')